# 🚀 ESP32-S3 TinyStories on Google Colab (Free Tier)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nicholaswilde/esp32-sandbox/blob/main/projects/s3-tiny-stories/s3_tiny_stories_colab.ipynb)

This notebook builds, trains, and quantizes **TinyStories language models** for on-device inference on the **ESP32-S3** microcontroller (16MB Flash, Octal PSRAM).

> **💡 Google Colab Free Tier Instructions:**
> 1. Ensure you are using the Free Tier T4 GPU runtime: Navigate to **Runtime** > **Change runtime type**.
> 2. Select **T4 GPU** under Hardware accelerator and click **Save**.

## 1. Environment & Hardware Verification

In [ ]:
# Verify GPU accelerator (T4 Free Tier)
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (Free Tier)")

## 2. Setup Workspace & Dependencies

In [ ]:
import os

# Clone repository if not already present
if not os.path.exists("esp32-sandbox"):
    !git clone https://github.com/nicholaswilde/esp32-sandbox.git

%cd /content/esp32-sandbox/projects/s3-tiny-stories

# Install required Python dependencies
!pip install -q tokenizers huggingface_hub requests numpy

## 📦 Track 1: Fast Verification with Pre-trained PLE TinyLM (28.9M params)

Fetch and verify the pre-trained 4-bit INT4 PLE model from `slvDev/esp32-ai-tinystories` (~14.9 MB):
This lets you test the exact weights and vocabulary configured for the ESP32-S3 without training.

In [ ]:
# Fetch pre-built model and verify SHA-256 hashes
%cd /content/esp32-sandbox/projects/s3-tiny-stories
!./scripts/fetch_model.sh

# Generate tokenizer decode table header (src/generated/vocab.h)
!python pc_tools/generate_vocab.py


In [ ]:
# Verify binary file size and PLE header
import os, struct
bin_path = "pc_tools/model.bin"
size_mb = os.path.getsize(bin_path) / (1024 * 1024)
print(f"Binary verified: {bin_path} ({size_mb:.2f} MB)")

with open(bin_path, "rb") as f:
    header = f.read(56)
    magic, ver, hbytes, flags = struct.unpack("<IIII", header[:16])
    vocab, out_vocab = struct.unpack("<II", header[16:24])
    print(f"Header check: Magic={magic:#x} ('PLE\\0'), Version={ver}, Vocab={vocab}, OutVocab={out_vocab}")


## 🧠 Track 2: Train Custom TinyLM PLE Model from Scratch (Free T4 GPU)

Train a custom Per-Layer Embedding (PLE) transformer with PyTorch on the T4 GPU:
1. **Prepare Data & Tokenizer**: Downloads the TinyStories dataset slice (~300MB) and trains a BPE tokenizer.
2. **Train Model**: Runs micro-batched training with `--arm ple` on GPU.
3. **Export to INT4**: Exports the trained checkpoint into the packed PLE binary format for ESP32 inference.

In [ ]:
# Prepare dataset slice and BPE tokenizer
%cd /content/esp32-sandbox/projects/s3-tiny-stories
!python -m research.tinystories.prepare --vocab 32768


In [ ]:
# Option A: Quick Verification Run (1.5M parameters, 500 steps, ~1-2 minutes on T4 GPU)
!python -m research.tinystories.train --arm ple --vocab 32768 --target-core 1500000 --steps 500 --seed 0 --tag v32768_c1500000 --micro-batch-size 8

# Option B: Full Training Run (28.9M parameters, 10,000 steps, ~30-45 minutes on T4 GPU)
# Uncomment to run full training:
# !python -m research.tinystories.train --arm ple --vocab 32768 --steps 10000 --seed 0


In [ ]:
# Export the trained checkpoint to PLE binary format
# For 1.5M test model:
!python -m research.tinystories.export --tokenizer data/tinystories/vocab-32768/tokenizer.json ple-v32768_c1500000-s0

# For full 28.9M model (if trained):
# !python -m research.tinystories.export --tokenizer data/tinystories/vocab-32768/tokenizer.json ple-s0


## 🧪 3. Test Text Generation in Colab

In [ ]:
# Sample text generation from the trained checkpoint
!python -m research.tinystories.sample \
  --tokenizer data/tinystories/vocab-32768/tokenizer.json \
  --run runs/ple-v32768_c1500000-s0.pt \
  --prompt "Once upon a time, there was a little robot"


## 💾 4. Download Model Artifacts for Flashing to ESP32-S3

In [ ]:
from google.colab import files
import glob, os

print("Downloading model artifacts to your local machine...")

# Pre-trained model if fetched
if os.path.exists("pc_tools/model.bin"):
    print("Downloading pc_tools/model.bin...")
    files.download("pc_tools/model.bin")

# Custom trained exported models
for model_file in glob.glob("artifacts/tinystories/*.bin"):
    print(f"Downloading {model_file}...")
    files.download(model_file)

print("\nOnce downloaded, place model.bin in 'projects/s3-tiny-stories/pc_tools/' and flash with:")
print("task flash-model")
